In [ ]:
# Accept parameters passed from orchestration notebook via dbutils.notebook.run()
# These simulate DAB variables in the bundle deployment

try:
    # Get parameters from dbutils.widgets (passed by dbutils.notebook.run)
    target_catalog = dbutils.widgets.get("catalog_name")  #staging_catalog in staging
    schema_prefix = dbutils.widgets.get("schema_prefix") #bronze in staging
    print(f"Using parameters from orchestration:")
    print(f"  catalog_name: {catalog_name}")
    print(f"  schema_prefix: {schema_prefix}")
except Exception:
    # Fallback to default values if not called from orchestration
    catalog_name = "dev_catalog"
    schema_prefix = "brz_raw_hrs"
    print(f"Using default values (not called from orchestration):")
    print(f"  catalog_name: {catalog_name}")
    print(f"  schema_prefix: {schema_prefix}")

In [ ]:
from pathlib import Path

#schema_prefix = "brz_raw_hrs"
#source_catalog = "dev_catalog"
#target_catalog = "staging_catalog"  # Or pass as parameter

# Read and execute SQL
sql_path = Path("../../sql/ddl/copy_hrs_data_from_source_to_target.sql")
sql_text = sql_path.read_text()

#print("source catalog ", source_catalog)
print("target catalog ", target_catalog)

# Drop existing target table if it exists (idempotent operation for DAB)
#target_table = f"{target_catalog}.brz_raw_hrs.randhrs1992_2022v1"
#spark.sql(f"DROP TABLE IF EXISTS {target_table}")
#print(f"  Dropped existing table (if any): {target_table}")

# Split and execute statements with parameter binding
statements = [stmt.strip() for stmt in sql_text.split(';') if stmt.strip()]

for i, stmt in enumerate(statements, 1):
    print(f"  Executing statement {i}/{len(statements)}")
    result = spark.sql(stmt, args={"target_catalog": target_catalog, "schema_prefix": schema_prefix})
    display(result)